# 01 - One-compartment pharmacokinetics

In this notebook we simulate how a drug's plasma concentration changes over time using the
simplest PK model: the body as a single, well-mixed compartment.

**Goals**
1. IV bolus: mono-exponential decay and half-life
2. Oral dosing: absorption, `Cmax` and `tmax`
3. Repeated dosing: accumulation and steady state

> Parameters are illustrative textbook-style values, **not** clinical guidance.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pkpd import models as m
from pkpd.drugs import DRUGS

trapz = getattr(np, "trapezoid", None) or np.trapz
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Drug parameters

In [ ]:
rows = []
for d in DRUGS.values():
    rows.append({
        "Drug": d.name, "V (L)": d.V, "CL (L/h)": d.CL, "k (1/h)": round(d.k, 3),
        "t1/2 (h)": round(d.t_half, 2), "F": d.F, "ka (1/h)": d.ka,
    })
pd.DataFrame(rows).set_index("Drug")

## 2. IV bolus

After an intravenous bolus the drug is instantly distributed, so
`C(t) = (Dose / V) * exp(-k t)`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, d in zip(axes, DRUGS.values()):
    t = np.linspace(0, 6 * d.t_half, 400)
    c = m.one_comp_iv(t, d.iv_dose, d.V, d.k)
    ax.plot(t, c, lw=2)
    ax.axvline(d.t_half, ls="--", c="gray")
    ax.set_title(f"{d.name}: {d.iv_dose:g} mg IV")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
plt.tight_layout()
plt.show()

### Sanity check: the half-life
After exactly one half-life, the concentration must be half of `C0`.

In [ ]:
for d in DRUGS.values():
    c0 = m.one_comp_iv(0.0, d.iv_dose, d.V, d.k)
    c_half = m.one_comp_iv(d.t_half, d.iv_dose, d.V, d.k)
    assert np.isclose(c_half, c0 / 2)
    print(f"{d.name:<12} C0 = {float(c0):7.3f} mg/L   C(t1/2) = {float(c_half):7.3f} mg/L")

## 3. Oral absorption

With first-order absorption the curve rises, peaks at `tmax`, then declines.
Only a fraction `F` of the dose reaches the circulation.

In [ ]:
d = DRUGS["paracetamol"]
t = np.linspace(0, 12, 500)
c_iv = m.one_comp_iv(t, d.iv_dose, d.V, d.k)
c_po = m.one_comp_oral(t, d.oral_dose, d.V, d.k, d.ka, d.F)

tmax = m.tmax_oral(d.k, d.ka)
cmax = m.cmax_oral(d.oral_dose, d.V, d.k, d.ka, d.F)

plt.figure(figsize=(7, 4))
plt.plot(t, c_iv, label="IV bolus", lw=2)
plt.plot(t, c_po, label="Oral", lw=2)
plt.scatter([tmax], [cmax], c="red", zorder=5, label=f"Cmax = {cmax:.1f} mg/L at {tmax:.2f} h")
plt.xlabel("Time (h)"); plt.ylabel("Concentration (mg/L)")
plt.title("Paracetamol 1000 mg: IV vs oral")
plt.legend(); plt.show()

### AUC check
The area under the oral curve should match `F * Dose / CL`.

In [ ]:
tt = np.linspace(0, 100, 100001)
cc = m.one_comp_oral(tt, d.oral_dose, d.V, d.k, d.ka, d.F)
print(f"AUC numerical : {trapz(cc, tt):.2f} mg.h/L")
print(f"AUC analytical: {m.auc_inf(d.oral_dose, d.CL, d.F):.2f} mg.h/L")

## 4. Repeated dosing and accumulation

Doses are superposed. The drug accumulates until input equals output (steady state).
The accumulation ratio is `R = 1 / (1 - exp(-k * tau))`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, d in zip(axes, DRUGS.values()):
    n = 8
    t = np.linspace(0, n * d.tau, 1500)
    c = m.multiple_dose(m.one_comp_oral, t, d.tau, n,
                        dose=d.oral_dose, V=d.V, k=d.k, ka=d.ka, F=d.F)
    css = m.css_average(d.oral_dose, d.CL, d.tau, d.F)
    ax.plot(t, c, lw=2)
    ax.axhline(css, ls="--", c="red", label=f"Css,avg = {css:.2f} mg/L")
    ax.set_title(f"{d.name}: {d.oral_dose:g} mg every {d.tau:g} h")
    ax.set_xlabel("Time (h)"); ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
plt.tight_layout(); plt.show()

print("Accumulation ratios:")
for d in DRUGS.values():
    print(f"  {d.name:<12} R = {m.accumulation_ratio(d.k, d.tau):.2f}")

## Takeaways
- The half-life depends on both clearance and volume: `t1/2 = ln(2) * V / CL`.
- Oral exposure is scaled by bioavailability `F`; `tmax` depends on `ka` and `k`.
- A short half-life relative to the dosing interval means little accumulation.

**Next:** notebook 02 adds a second (peripheral) compartment.